# Explore a NewsMead gaze recording

This notebook opens the **actual saved study packages**. It checks the selected
recording, builds compact tables and plots what was recorded. The default is the
earlier `storage-check` performance visit; the later popup attempt stays separate.

This is exploratory engineering work. These visits are not experimental groups,
and gaze samples or overlapping R/D/F windows are not independent participants.
There are no significance tests or claims of reading benefit here.

Use **Run → Run All Cells**. Change the paths/IDs in the settings cell for another
recording. The original files are read only. Generated CSVs, figures and receipts
go to a new directory under the project's ignored `output/` folder.

## What is inside the saved data?

| Family | Useful contents | Interpretation |
| --- | --- | --- |
| Session/package | Participant, installation, run, article-reading and transport IDs; settings; checksums | Each article visit is separate; provider restarts can be linked segments |
| Gaze | Capture/delivery clocks, calibrated pre-smoothing `raw` x/y, smoothed `selected` x/y, source reason, context flags | Model estimates in screen pixels; no camera images and no independent accuracy measurement |
| R/D/F | Native window bounds, regression rate, dwell fraction, line-stay rate, quality and native MD/MD² | The code's `fixation_rate_per_minute` currently describes line stays; it is not a validated count of ocular fixations |
| Scanpath | Native outputs, producer/generation, event and pattern IDs | Revisions, fixations and pattern events are different record types; P13 includes boundaries |
| Aids | Chosen/cleared actions, offers, acceptance, consultation and layout proxies | A layout proxy does not prove that content was seen or a definition loaded successfully |
| Baseline/reference | Frozen R/D/F profile and source attempt; baseline fit information | Coordinate-MD needs its own compatible reference; this notebook does not create one |
| Health/boundaries | Missingness, pauses, context transitions, queue losses, recovery/close state | An intact file does not make every observation eligible or accurate |

The helper makes **compact projections**, not another canonical raw archive.
All original native payloads remain in the source packages. Large layout/history
blobs are deliberately absent from these working tables.

In [ ]:
from pathlib import Path
import sys, os, json, platform, hashlib, uuid
from datetime import datetime, timezone

def find_repo():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "tools/gaze/study_package.py").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the NewsMead project, or set REPO_ROOT explicitly.")

REPO_ROOT = find_repo()
DERIVED_BASE = REPO_ROOT / "output/study-notebook"
os.environ.setdefault("MPLCONFIGDIR", str(DERIVED_BASE / "matplotlib-cache"))
sys.path.insert(0, str(REPO_ROOT / "tools/gaze"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, Markdown
import study_notebook as sn

pd.set_option("display.max_columns", 16)
pd.set_option("display.max_colwidth", 85)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "axes.spines.top": False, "axes.spines.right": False})

# A parent directory of packages, one package directory, or an exported ZIP.
SOURCE = REPO_ROOT / "output/study-live-check-04/packages"
SELECTED_SESSION_ID = "f22986df-be2f-494a-bc5b-2ef0ebc00261"
# Optional second visit, inspected separately. Set to None for a different dataset.
POPUP_SESSION_ID = "60bb4931-432b-4f27-85db-0718b23cb3d4"
WRITE_EXPORTS = True

if not SOURCE.exists():
    raise FileNotFoundError(f"Set SOURCE to your copied study ZIP or package directory: {SOURCE}")
source_resolved = SOURCE.resolve()
if DERIVED_BASE.resolve() == source_resolved or (SOURCE.is_dir() and DERIVED_BASE.resolve().is_relative_to(source_resolved)):
    raise ValueError("DERIVED_BASE must be outside the source recording directory.")

RUN_OUTPUT = DERIVED_BASE / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
if WRITE_EXPORTS:
    RUN_OUTPUT.mkdir(parents=True)
versions = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
            "matplotlib": __import__("matplotlib").__version__, "adapter": sn.VERSION}
display(versions)
print("Source:", SOURCE)
print("Derived output:", RUN_OUTPUT)

## 1. Choose a recording, then verify it

The catalog below reads **manifest declarations only**. `DECLARED_UNVERIFIED`
does not establish integrity. Loading a selected session invokes the existing
strict importer: stored hashes, gzip decoding, counts, identities, sequence/loss
metadata and available late receipts are checked.

Choose a **transport session ID**, not an article ID. The article-reading ID
groups linked segments; reopening the same article can create another reading ID.
For a multi-package ZIP, the optional `study_transfer_check.py` command also checks
that every package declared by the transfer inventory arrived (see README).

In [ ]:
catalog = pd.DataFrame(sn.catalog_sources(SOURCE))
if catalog.empty:
    raise ValueError("No package manifests found in SOURCE.")
display(catalog[["package_id", "session_id", "participant_id", "status", "content_complete",
                 "record_count", "stored_bytes", "decoded_bytes", "validation"]])

recording = sn.load_recording(SOURCE, session_id=SELECTED_SESSION_ID)
metadata, receipt = recording.metadata, recording.receipt
frames = {name: pd.DataFrame(rows) for name, rows in recording.tables.items()}
display(pd.Series({key: metadata.get(key) for key in ["validation", "participant_id", "phase", "condition",
    "article_id", "article_reading_id", "session_id", "start_ms", "end_ms", "recorded_span_s",
    "stored_bytes", "decoded_bytes", "coordinate_md_status"]}, name="selected recording"))
display(pd.Series(receipt["table_counts"], name="compact table rows"))
display({"manifest_sha256": receipt["manifest_sha256"], "content_complete": receipt["content_complete"],
         "declared_losses": receipt["declared_losses"], "unknown_tail": receipt["unknown_tail"],
         "annotations": receipt["annotations"], "late_receipt_source": receipt["late_receipt_source"]})

## 2. Inspect clocks, missingness and context

`capture_ms` and `delivery_ms` are absolute elapsed milliseconds in the declared
clock domain. Derived offsets below use this session's start. UTC anchors are
separate metadata; do not merge these clocks with the app's separate usage-log
relative clock or across different clock domains without an explicit mapping.

Missing numbers stay missing. **Zero is a real numeric value**, including MD=0;
never replace unavailable scores with zero. `context_allowed` is a collector gate,
not tracking accuracy or confirmation that the participant was reading.

In [ ]:
gaze = frames["gaze"].copy()
if gaze.empty:
    raise ValueError("This package contains no gaze observations.")
for column in ["capture_ms", "delivery_ms", "raw_x_px", "raw_y_px", "selected_x_px", "selected_y_px", "latency_ms"]:
    if column in gaze:
        gaze[column] = pd.to_numeric(gaze[column], errors="raise")
display(gaze[[c for c in ["seq", "provider_epoch_id", "delivery_id", "capture_offset_s", "delivery_offset_s",
                          "reason", "context_allowed", "raw_x_px", "raw_y_px", "selected_x_px", "selected_y_px", "latency_ms"] if c in gaze]].head(10))
display(gaze.groupby(["reason", "context_allowed"], dropna=False).size().rename("observations").to_frame())
display(gaze[[c for c in ["capture_ms", "raw_x_px", "raw_y_px", "selected_x_px", "selected_y_px"] if c in gaze]].isna().sum().rename("missing values"))

start_ms, end_ms = metadata["start_ms"], metadata["end_ms"]
if start_ms is None or end_ms is None or end_ms <= start_ms:
    raise ValueError("A known positive logical session span is required for the time plots. Inspect recovery/close state first.")
capture_in_bounds = gaze["capture_ms"].between(start_ms, end_ms, inclusive="both")
source_coordinate = gaze["reason"].eq("coordinate") & capture_in_bounds
allowed_coordinate = source_coordinate & gaze["context_allowed"].eq(True)
latencies = gaze.loc[source_coordinate, "latency_ms"].dropna()
diagnostics = {
    "recorded_span_s_including_exclusions": metadata["recorded_span_s"],
    "coordinate_outputs_in_bounds": int(source_coordinate.sum()),
    "coordinate_outputs_context_allowed": int(allowed_coordinate.sum()),
    "missing_capture_observations": int(gaze["capture_ms"].isna().sum()),
    "timestamped_captures_outside_bounds": int((gaze["capture_ms"].notna() & ~capture_in_bounds).sum()),
    "median_coordinate_latency_ms": float(latencies.median()) if len(latencies) else None,
    "p95_coordinate_latency_ms_linear": float(latencies.quantile(.95)) if len(latencies) else None,
    "negative_coordinate_latency_count": int((latencies < 0).sum()),
}
display(pd.Series(diagnostics, name="descriptive checks"))

## 3. Recorded output rate and latency

Rates are counts per **recorded elapsed second in each bin**; the final bin uses
its actual shorter duration. Startup and excluded periods remain in the
denominator. The allowed-coordinate curve shows a different numerator, not an
"accuracy-adjusted FPS". These are not independently measured camera/screen FPS.
Latency percentiles in the bins use all finite capture/delivery pairs in the bin;
the summary above restricts latency to native coordinate outputs.

In [ ]:
cadence = pd.DataFrame(sn.cadence_bins(recording.tables["gaze"], start_ms, end_ms, bin_seconds=10))
cadence["offset_s"] = (cadence["start_ms"] - start_ms) / 1000
display(cadence[["offset_s", "span_s", "coordinate_count", "coordinate_hz", "coordinate_context_allowed_count",
                 "coordinate_context_allowed_hz", "median_latency_ms", "p95_latency_ms"]])

figures = {}
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, constrained_layout=True)
axes[0].plot(cadence.offset_s, cadence.coordinate_hz, "o-", label="Coordinate outputs / recorded second")
axes[0].plot(cadence.offset_s, cadence.coordinate_context_allowed_hz, "s--", label="Context-allowed coordinate outputs / recorded second")
axes[0].set_ylabel("Outputs / s"); axes[0].legend(fontsize=8); axes[0].grid(alpha=.2)
axes[1].plot(cadence.offset_s, cadence.median_latency_ms, "o-", label="Median")
axes[1].plot(cadence.offset_s, cadence.p95_latency_ms, "s--", label="95th percentile")
axes[1].set(xlabel="Seconds since logical session start", ylabel="Capture-to-delivery age (ms)")
axes[1].legend(); axes[1].grid(alpha=.2)
fig.suptitle("Recorded gaze cadence and latency · " + metadata["session_id"][:8])
figures["cadence_latency"] = fig
plt.show()

## 4. Raw and selected gaze positions

The two coordinate streams are shown separately. Screen origin is at the upper
left; y increases downward. The rectangle is the recorded physical screen, not
the article's text bounds. Points outside it are retained. No interpolation,
outlier clipping or new fixation detector is applied. Scatter is a position
distribution, not proof of a reading scanpath or word-level gaze accuracy.

In [ ]:
dimensions = [d for d in metadata.get("screen_dimensions", []) if d.get("width_px") and d.get("height_px")]
if len(dimensions) == 1:
    width, height = dimensions[0]["width_px"], dimensions[0]["height_px"]
    fig, axes = plt.subplots(1, 2, figsize=(9, 7), constrained_layout=True)
    for ax, prefix, label in zip(axes, ["raw", "selected"], ["Calibrated pre-smoothing output", "Selected / smoothed output"]):
        x, y = gaze[prefix + "_x_px"], gaze[prefix + "_y_px"]
        finite_points = source_coordinate & np.isfinite(x) & np.isfinite(y)
        allowed = finite_points & gaze.context_allowed.eq(True)
        flagged = finite_points & ~gaze.context_allowed.eq(True)
        ax.scatter(x[flagged], y[flagged], s=9, c="#b8bec8", alpha=.55, label="Excluded context")
        scatter = ax.scatter(x[allowed], y[allowed], c=gaze.loc[allowed, "capture_offset_s"], s=7,
                             cmap="viridis", alpha=.65, label="Allowed context")
        ax.add_patch(Rectangle((0, 0), width, height, fill=False, edgecolor="#1f2937", linewidth=1.2))
        ax.set(xlabel="Screen x (px)", ylabel="Screen y (px)", title=label)
        ax.invert_yaxis(); ax.set_aspect("equal", adjustable="datalim"); ax.legend(fontsize=8)
        fig.colorbar(scatter, ax=ax, label="Capture offset (s)", shrink=.65)
    figures["gaze_positions"] = fig
    plt.show()
else:
    print("Select one physical-screen context before plotting positions; recorded dimensions:", dimensions)

## 5. Native R/D/F features and score availability

The curves below use **recorded native outputs**, not a reimplementation.
An R/D/F window is normally about 10 seconds with updates around every 0.5 seconds,
so adjacent windows overlap. A window count is not a participant sample size.
Feature-quality flags and MD availability are distinct.

For MD, only rows whose native status is `AVAILABLE` are plotted. Other statuses
remain in the table and availability counts; they are not converted to MD=0.
No "high instability" cutoff or benefit interpretation is introduced. A larger
MD means greater departure from this stored reference in its feature space; it
does not by itself establish difficulty or harm. Rereading can be purposeful.

In [ ]:
rdf = frames["rdf"]
ticks = rdf.loc[rdf["kind"].eq("rdf_tick")].copy() if not rdf.empty else pd.DataFrame()
if not ticks.empty:
    ticks["offset_s"] = (pd.to_numeric(ticks["feature_vector_timestamp_ms"], errors="raise") - start_ms) / 1000
    display(ticks[[c for c in ["offset_s", "feature_vector_window_start_ms", "feature_vector_window_end_ms",
        "feature_vector_quality", "feature_vector_valid_fraction", "feature_vector_observed_coverage_fraction",
        "score_status", "score_md", "score_md2"] if c in ticks]].head(12))
    display(ticks.score_status.value_counts(dropna=False).rename("native score ticks").to_frame())
    fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True, constrained_layout=True)
    feature_columns = [
        ("feature_vector_values_regression_rate_per_minute", "Native regressions / min"),
        ("feature_vector_values_dwell_fraction", "Native dwell fraction"),
        ("feature_vector_values_fixation_rate_per_minute", "Native line stays / min"),
    ]
    native_valid = ticks.feature_vector_quality.eq("VALID")
    for ax, (column, label) in zip(axes[:3], feature_columns):
        values = pd.to_numeric(ticks[column], errors="raise")
        ax.scatter(ticks.loc[~native_valid, "offset_s"], values[~native_valid], s=10, c="#b8bec8", label="Other native quality")
        ax.scatter(ticks.loc[native_valid, "offset_s"], values[native_valid], s=8, c="#2563eb", label="Native quality VALID")
        ax.set_ylabel(label); ax.grid(alpha=.2)
    axes[0].legend(fontsize=8)
    available = ticks.score_status.eq("AVAILABLE")
    axes[3].plot(ticks.offset_s, pd.to_numeric(ticks.score_md, errors="raise").where(available), ".-", color="#7c3aed", markersize=3)
    axes[3].set(xlabel="Seconds since logical session start", ylabel="Native R/D/F MD (dimensionless)")
    axes[3].grid(alpha=.2)
    fig.suptitle("Recorded R/D/F windows and Mahalanobis distance")
    figures["native_rdf_md"] = fig
    plt.show()
else:
    print("No native rdf_tick scores in this recording. Baseline windows are a different record family:")
    if not rdf.empty:
        display(rdf.kind.value_counts().to_frame("records"))

references = frames["references"]
display(pd.DataFrame(metadata.get("source_reference_attempts", [])))
if not references.empty:
    display(references[[c for c in ["kind", "record_id", "baseline_attempt_id", "fit_status", "reference_session_id",
                                   "reference_participant_code", "participant_match"] if c in references]])
    profile_cells = references.get("reference_profile_json", pd.Series(dtype="object")).dropna()
    if len(profile_cells) and profile_cells.iloc[0] != "null":
        frozen_profile = json.loads(profile_cells.iloc[0])
        display(pd.Series(frozen_profile.get("mean", {}), name="frozen R/D/F reference means"))
        order = ["regression_rate_per_minute", "dwell_fraction", "fixation_rate_per_minute"]
        covariance = frozen_profile.get("covariance", {})
        if all(f"m{i}{j}" in covariance for i in range(3) for j in range(3)):
            display(pd.DataFrame([[covariance[f"m{i}{j}"] for j in range(3)] for i in range(3)],
                                 index=order, columns=order).rename_axis("stored covariance"))
print("Coordinate-MD:", metadata["coordinate_md_status"], "— no reference fitted or substituted by this notebook.")

## 6. Native scanpath outputs

Count only the direct `ScanpathEvent` outputs from the `pattern` producer here.
Per-consumer outputs, candidate revisions and layout records remain separate.
The counts are **recorded event rows**, not inferred fixation counts. P13 contains
boundaries, while P12 includes residence/progress summaries. Neither should be
read as a count of independent reading behaviours. Word and line resolution are
shown separately because they can describe the same underlying activity.

IDs are local to their session/producer/generation. Inspect repeated scoped IDs
before deciding how an eventual event-level analysis should handle them.

In [ ]:
scanpath = frames["scanpath"]
patterns = pd.DataFrame()
if not scanpath.empty and "producer" in scanpath and "output_type" in scanpath:
    patterns = scanpath.loc[scanpath.kind.eq("scanpath_output") & scanpath.producer.eq("pattern") &
                            scanpath.output_type.eq("direct_pattern_event")].copy()
if not patterns.empty:
    scope = [c for c in ["package_id", "session_id", "producer", "generation", "output_resolution", "output_event_id"] if c in patterns]
    print("Rows with repeated scoped event IDs:", int(patterns.duplicated(scope, keep=False).sum()))
    display(patterns[[c for c in ["record_id", "producer", "generation", "output_event_id", "output_pattern_id",
                                "output_resolution", "output_start_ms", "output_end_ms", "source_elapsed_ms"] if c in patterns]].head(12))
    resolution = patterns.get("output_resolution", pd.Series(index=patterns.index, dtype="object")).fillna("not recorded")
    count_view = patterns.assign(resolution_label=resolution)
    counts = count_view.groupby(["output_pattern_id", "resolution_label"], dropna=False).size().unstack(fill_value=0).sort_index()
    display(counts)
    if "output_metrics_reason" in patterns:
        display(count_view.loc[count_view.output_pattern_id.eq("P13")].groupby(
            ["resolution_label", "output_metrics_reason"], dropna=False).size().rename("boundary output records").to_frame())
    if "output_metrics_kind" in patterns:
        display(count_view.loc[count_view.output_pattern_id.eq("P12")].groupby(
            ["resolution_label", "output_metrics_kind"], dropna=False).size().rename("P12 summary output records").to_frame())
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
    counts.plot.bar(stacked=True, ax=axes[0])
    axes[0].set(xlabel="Native pattern ID", ylabel="Recorded event rows", title="All outputs · P12 summaries / P13 boundaries")
    other = counts.loc[~counts.index.isin(["P12", "P13"])]
    if not other.empty:
        other.plot.bar(stacked=True, ax=axes[1])
    axes[1].set(xlabel="Native pattern ID", ylabel="Recorded event rows", title="Other output IDs · enlarged view")
    figures["native_pattern_outputs"] = fig
    plt.show()
else:
    print("No direct native pattern outputs in this recording.")

## 7. Aid decisions and the separate popup attempt

CHOSEN is a policy decision. An offer shown, user acceptance, consultation and a
panel-layout proxy are subsequent distinct events. Keep their clocks and action
IDs, and do not report layout proxies as measured human-visible exposure.

The optional second session below is the later deliberate popup attempt. Its
results are shown separately; it is not combined with the performance visit or
treated as another participant/condition. Set `POPUP_SESSION_ID=None` to inspect
only the primary visit.

In [ ]:
aids = frames["aids"]
decisions = aids.loc[aids.kind.eq("aid_decision")].copy() if not aids.empty else pd.DataFrame()
if not decisions.empty:
    grouping = ["decision_status", "decision_action_aid_type"]
    decision_counts = decisions.reindex(columns=grouping).groupby(grouping, dropna=False).size().rename("decision records").to_frame()
    display(decision_counts)

secondary = None
if POPUP_SESSION_ID is not None and POPUP_SESSION_ID != SELECTED_SESSION_ID:
    secondary = sn.load_recording(SOURCE, session_id=POPUP_SESSION_ID)
    display(pd.Series({k: secondary.metadata.get(k) for k in ["session_id", "article_reading_id", "article_id", "recorded_span_s"]}, name="separate popup attempt"))
event_recording = secondary if secondary is not None else recording
event_aids = pd.DataFrame(event_recording.tables["aids"])
event_start = event_recording.metadata["start_ms"]
event_rows = []
if not event_aids.empty:
    for _, row in event_aids.iterrows():
        label = None
        if row["kind"] == "aid_runtime_definition_offer":
            label = "offer: " + str(row.get("event"))
        elif row["kind"] == "aid_runtime_definition_consultation":
            active = row.get("active")
            label = ("consultation starts" if bool(active) else "consultation ends") if isinstance(active, (bool, np.bool_)) else "consultation state unknown"
        elif row["kind"] == "aid_runtime_render" and row.get("presentation_phase") in ["lookup_offer", "definition_panel"]:
            label = str(row["presentation_phase"]) + " layout proxy"
        elif row["kind"] == "aid_decision" and row.get("decision_action_aid_type") == "WORD_DEFINITION":
            label = "decision: " + str(row.get("decision_status"))
        if label is not None:
            event_rows.append({"package_id": row["package_id"], "session_id": row["session_id"],
                "clock_domain_id": row["clock_domain_id"], "record_id": row["record_id"], "seq": row["seq"], "label": label,
                "source_elapsed_ms": row["source_elapsed_ms"], "offset_s": (row["source_elapsed_ms"]-event_start)/1000,
                "action_id": row.get("action_id") if pd.notna(row.get("action_id")) else row.get("decision_action_id"),
                "verified_visibility": row.get("verified_visibility")})
popup_events = pd.DataFrame(event_rows)
if not popup_events.empty:
    display(popup_events)
    labels = list(dict.fromkeys(popup_events.label))
    fig, ax = plt.subplots(figsize=(11, 4), constrained_layout=True)
    ax.scatter(popup_events.offset_s, [labels.index(x) for x in popup_events.label], s=50, color="#2563eb")
    ax.set_yticks(range(len(labels)), labels)
    ax.set(xlabel="Seconds since this popup session's logical start", title="Recorded lookup events — layout proxies are not measured visibility")
    ax.grid(axis="x", alpha=.2)
    figures["definition_events"] = fig
    plt.show()
else:
    print("No definition-offer/panel events recorded in the selected event session.")

## 8. Save compact working tables and provenance

Exports are new derived files. They keep original identity/clock fields and native
availability flags; the source package remains the canonical archive. CSV blank
numeric cells mean missing/unavailable, not zero. Focused nested fields have
`_json` columns. For the complete payloads use the existing full study importer.

One output folder is created per notebook run. The popup attempt receives its own
subfolder. The receipt identifies source manifests, software versions, table
counts and the explicit filters/denominators used in these plots.

In [ ]:
if WRITE_EXPORTS:
    primary_out = RUN_OUTPUT / "primary"
    primary_out.mkdir()
    for name, frame in frames.items():
        frame.to_csv(primary_out / f"{name}.csv", index=False, encoding="utf-8")
    cadence.to_csv(primary_out / "cadence_10s.csv", index=False, encoding="utf-8")
    if not patterns.empty:
        patterns.to_csv(primary_out / "direct_pattern_events.csv", index=False, encoding="utf-8")
    if secondary is not None:
        popup_out = RUN_OUTPUT / "separate_popup_attempt"
        popup_out.mkdir()
        for name in ["gaze", "rdf", "aids", "boundaries", "references"]:
            pd.DataFrame(secondary.tables[name]).to_csv(popup_out / f"{name}.csv", index=False, encoding="utf-8")
    if not popup_events.empty:
        popup_events.to_csv(RUN_OUTPUT / "definition_event_timeline.csv", index=False, encoding="utf-8")
    for name, figure in figures.items():
        figure.savefig(RUN_OUTPUT / f"{name}.png", bbox_inches="tight")
    export_receipt = {"notebook_workflow": "newsmead_exploratory_notebook_v1", "versions": versions,
        "source": str(SOURCE), "selected_session_id": SELECTED_SESSION_ID,
        "primary_metadata": metadata, "primary_verification": receipt,
        "secondary_metadata": secondary.metadata if secondary else None,
        "secondary_verification": secondary.receipt if secondary else None,
        "diagnostics": diagnostics, "cadence_bin_seconds": 10,
        "cadence_denominator": "actual recorded bin span including exclusions",
        "latency_quantile_method": "linear at (n-1)*q; no clipping",
        "gaze_position_plot": "native coordinate reason, finite raw/selected x/y, capture in logical bounds; context flags shown separately",
        "rdf_md_plot": "native score status AVAILABLE only; all statuses retained in CSV",
        "scanpath_plot": "recorded direct_pattern_event rows from producer pattern, split by word/line resolution; P12 includes summaries and P13 boundaries; no deduplication",
        "coordinate_md_computed": False, "inferential_tests_run": False,
        "source_files_modified": False}
    (RUN_OUTPUT / "notebook-receipt.json").write_text(json.dumps(export_receipt, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    print("Saved tables, figures and provenance to:", RUN_OUTPUT)
else:
    print("WRITE_EXPORTS=False — tables/figures are in memory only.")

## How to develop this into thesis analysis

1. Check complete/incomplete packages, tracking reasons, context exclusions and
   score availability before interpreting values. Keep a record of unsuccessful
   sessions, too.
2. Decide the experimental unit and planned participant/article-level outcomes
   before group comparisons. These engineering runs cannot establish aid benefit.
3. Preserve intended conditions, materials and participant/session identities.
   The label `compressed` describes this engineering run, not an aid assignment.
4. Coordinate-MD can be added once the reference workflow and compatibility are
   settled. Never fit each evaluated window to itself or invent missing scores.
5. Keep tuning/exploratory recordings separate from independent validation data.

The current full importer has a 512 MiB decoded-package budget. Loading remains
one verified package at a time and may require substantial RAM despite gzip's
small disk size. This notebook does not replace that limit with silent truncation.

## References and software notes

- [NewsMead local study collection workflow](../docs/study-collection-local-v1.md)
  and [live compressed-recording check](../docs/study-live-storage-recheck-2026-11-03.md).
- [Jupyter notebook format](https://nbformat.readthedocs.io/en/latest/format_description.html)
  and [NBClient execution](https://nbclient.readthedocs.io/en/latest/client.html).
- [pandas structured-data normalization](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html).
  The helper uses a narrower projection to keep large native context blobs out of working tables.
- Procedural exploratory-analysis guidance: Kassis, T., Agarwal, V., He, Y.,
  Patel, D., & Brueckner, A. M. (2026). *Scientific Agent Skills: A Library of
  Procedural Knowledge for Research Agents.* [arXiv:2609.00065](https://doi.org/10.48550/arXiv.2609.00065).
  This is a workflow reference, not evidence validating the gaze metrics.